# Train Orchestrator — Clasificador EfficientNet-B0 (5 repeticiones)

Orquesta el entrenamiento+testing del clasificador con el protocolo de **5 repeticiones**.
Para la repeticion `i` (1..5): entrena con `split_repetition_i.json` (seed `i`), guarda
`fs_weights_split_i.pth`, testea con su seccion `test` y guarda las predicciones + el
Grad-CAM de las predichas glaucoma.

**Antes de correr:** sube a tu Google Drive (a) la carpeta con las imagenes/mascaras/
anotaciones (carpetas `1209/`, `1217/`, ...) y (b) asegurate de que los 5
`split_repetition_*.json` esten en `Classifier/splits/` del repo.

## 1. Bootstrap (Colab + Drive)

In [ ]:
import os
REPO = "/content/Medgemma_Segmentation_CIARP_2026"
BRANCH = "Classifier-Implementation"
if not os.path.isdir(REPO):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b {BRANCH} https://github.com/TheBug95/Medgemma_Segmentation_CIARP_2026.git {REPO}
%cd {REPO}
!GIT_LFS_SKIP_SMUDGE=1 git pull origin {BRANCH}
from google.colab import drive
drive.mount('/content/drive')
import torch; print("GPU disponible:", torch.cuda.is_available())

## 2. Cargar config

Ajusta `data_root` a la carpeta de tu Drive con las imagenes.

In [ ]:
import sys, yaml
from pathlib import Path
CLASSIFIER_DIR = Path(REPO) / "Classifier"
sys.path.insert(0, str(CLASSIFIER_DIR))

with open(CLASSIFIER_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

# >>> AJUSTA esta ruta a la carpeta de tu Drive con las imagenes (carpetas 1209/, ...)
config["data"]["data_root"] = "/content/drive/MyDrive/GlaucomaData"

print("data_root:", config["data"]["data_root"])
print("splits:   ", config["data"]["split_files"])
print("seeds:    ", config["training"]["seeds"])
print("subsample:", config["training"]["train_subsample"])

## 3. (Opcional) Verificar datos

Confirma que se leen los splits y las etiquetas.

In [ ]:
from training.data_interface import load_split, build_samples
data_root = config["data"]["data_root"]
split1_path = CLASSIFIER_DIR / config["data"]["splits_dir"] / config["data"]["split_files"][0]
split1 = load_split(split1_path)
for sec in ["train", "validation", "test"]:
    print(sec, "->", len(split1.get(sec, [])))

# Leer etiquetas de las primeras 3 (requiere que data_root tenga las anotaciones)
samples = build_samples(split1["train"][:3], data_root,
                        config["data"]["label_field"], config["data"]["label_map"])
print("ejemplos (archivo, label):", [(p.name, l) for p, l in samples])

## 4. Correr las 5 repeticiones

**Puede tardar varios minutos** (5 entrenamientos).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
from training.few_shot import run_all_repetitions

summary = run_all_repetitions(config)
for rep in summary["repetitions"]:
    print(f"rep {rep['repetition']} (seed {rep['seed']}): "
          f"val_f1={rep['best_val_f1']:.3f} | test={rep['test']['n_test']} imgs "
          f"| gradcam={rep['test']['n_gradcam']} -> {Path(rep['checkpoint']).name}")

## 5. Inspeccionar una prediccion + Grad-CAM

In [ ]:
from efficientNet_classification import load_classifier, classify_image
import matplotlib.pyplot as plt

ckpt_dir = Path(config["paths"]["checkpoints_dir"])
ckpt_dir = ckpt_dir if ckpt_dir.is_absolute() else CLASSIFIER_DIR / ckpt_dir
classifier = load_classifier(ckpt_dir / "fs_weights_split_1.pth", config)

test_samples = build_samples(split1["test"], data_root,
                             config["data"]["label_field"], config["data"]["label_map"])
# Buscar una imagen glaucoma (label==1) del test
glaucoma_path = next((p for p, l in test_samples if l == 1), test_samples[0][0])
res = classify_image(glaucoma_path, classifier, config)
print("Prediccion:", res["prediction"], "| dist:", res["distribution"])
if res["gradcam"] is not None:
    plt.figure(figsize=(6, 6))
    plt.imshow(res["gradcam"]["overlay_image"]); plt.axis("off")
    plt.title(f"Grad-CAM | {res['prediction']}"); plt.show()

## 6. Guardar pesos y resultados en Drive

In [ ]:
import shutil, datetime
dst = "/content/drive/MyDrive/Classifier_results_" + datetime.datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs(dst, exist_ok=True)
for sub in ["checkpoints", "results"]:
    src = CLASSIFIER_DIR / sub
    if src.exists():
        shutil.copytree(src, Path(dst) / sub, dirs_exist_ok=True)
print("Copiado a Drive:", dst)